# Experimento: fuzzy connectedness vascular 3D

Cópia experimental do fluxo de `fuzzy.ipynb` para testar segmentação arterial por fuzzy connectedness.

A ideia é manter o pipeline normal até a detecção dos óstios e o cálculo da vesselness arterial. Depois disso, a segmentação da artéria pode ser feita por:

- `normal_region_growing`: baseline atual do pipeline;
- `fuzzy_connectedness_*`: propagação fuzzy max-min a partir dos óstios.

As saídas principais são: status dos óstios, Dice por imagem/abordagem e resumo médio por abordagem.


## 1. Ambiente

Configura o caminho do repositório e importa as funções usadas no experimento.


In [1]:
# ruff: noqa: E402
import sys
from pathlib import Path

CURRENT_DIR = Path.cwd().resolve()
REPO_ROOT = CURRENT_DIR if (CURRENT_DIR / "src").exists() else CURRENT_DIR.parent
SRC_DIR = REPO_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

REPO_ROOT



PosixPath('/home/matheus/workspace/coronary-centerline-detection')

In [2]:
import copy

import numpy as np
import pandas as pd

from utils.project.config import load_config_json, scale_config_to_resolution
from utils.project.dataset import get_data_splits
from utils.project.notebook_env import resolve_imagecas_base_path
from utils.processing import (
    downscale_image_ndi,
    threshold_image_with_offset,
)
from utils.segmentation import (
    build_background_seeds,
    build_lcc_image_from_mask,
    collect_local_object_seeds,
    detect_and_evaluate_ostia,
    fuzzy_connectedness_segmentation,
    get_or_compute_vesselness,
    get_or_detect_aorta_circles,
    get_or_segment_aorta,
    limit_candidate_mask_by_vesselness,
    normal_region_growing_from_ostia,
    postprocess_artery_mask,
    valid_seed,
)
from utils.utils import dice_score, normalize_vesselness
from utils.utils.nifti_io import load_raw_img_and_label


## 2. Parâmetros

Os padrões abaixo testam uma connectedness mais permissiva e mais eficiente: `finalized` na heap, multi-seeds locais, `alpha` menor, `sigma_HU` maior, piso de vesselness e combinação ponderada entre vesselness e HU.


In [3]:
CONFIG_PATH = REPO_ROOT / "config/pipeline_config.json"
BASE_PATH = resolve_imagecas_base_path()
BASE_SAVE_PATH = REPO_ROOT / "output" / "tmp_fuzzy_connectedness"

TRAIN_SAMPLE_SIZE = 30
VAL_SAMPLE_SIZE = 0
DOWNSCALE_FACTORS = (2, 2, 1)
MIN_HU = -300

# Região candidata padrão para a propagação. Variantes podem sobrescrever esse valor.
CANDIDATE_MIN_VESSELNESS = 0.01
MAX_CANDIDATE_VOXELS = 500_000
MAX_PROCESSED_VOXELS = 500_000

# Sementes locais ao redor dos óstios para reduzir dependência de um único voxel.
USE_LOCAL_MULTI_SEEDS = True
SEED_SEARCH_RADIUS = 3
MAX_SEEDS_PER_OSTIUM = 8
SEED_MIN_VESSELNESS = 0.02
MIN_SEED_DISTANCE_VOXELS = 1.0

USE_BACKGROUND_SEEDS = False
BACKGROUND_SEED_MAX_COUNT = 300
BACKGROUND_SEED_LOW_VESSELNESS_PERCENTILE = 10.0

BEST_ALPHA = 0.16
BEST_SIGMA_HU = 100.0
BEST_VESSELNESS_WEIGHT = 0.90
BEST_CANDIDATE_MIN_VESSELNESS = 0.020

_FC_WEIGHTED_PRODUCT_BASE = {
    "alpha": BEST_ALPHA,
    "sigma_hu": BEST_SIGMA_HU,
    "neighborhood": 18,
    "vesselness_affinity_mode": "geometric_mean",
    "vesselness_power": 1.00,
    "vesselness_floor": 0.02,
    "edge_affinity_mode": "weighted_product",
    "vesselness_weight": BEST_VESSELNESS_WEIGHT,
    "candidate_min_vesselness": BEST_CANDIDATE_MIN_VESSELNESS,
    "variant_group": "selected_top5_30_images",
}

FUZZY_CONNECTEDNESS_VARIANTS = {}


def add_fc_variant(name: str, **overrides) -> None:
    """Adiciona uma variante independente do RG normal."""
    params = {**_FC_WEIGHTED_PRODUCT_BASE, **overrides}
    FUZZY_CONNECTEDNESS_VARIANTS[name] = params


# Cinco melhores FC da rodada com 10 imagens, mantendo normal_region_growing como baseline.
add_fc_variant(
    "fc_wp_seed_s100_r2_m4_seed020_n18",
    seed_search_radius=2,
    max_seeds_per_ostium=4,
    seed_min_vesselness=0.020,
)
add_fc_variant(
    "fc_wp_seed_s100_r2_m6_seed020_n18",
    seed_search_radius=2,
    max_seeds_per_ostium=6,
    seed_min_vesselness=0.020,
)
add_fc_variant(
    "fc_wp_seed_s100_r3_m4_seed020_n18",
    seed_search_radius=3,
    max_seeds_per_ostium=4,
    seed_min_vesselness=0.020,
)
add_fc_variant(
    "fc_wp_seed_s100_r2_m8_seed020_n18",
    seed_search_radius=2,
    max_seeds_per_ostium=8,
    seed_min_vesselness=0.020,
)
add_fc_variant(
    "fc_wp_alpha_a160_s100_w090_c020_n18",
    variant_group="selected_top5_anchor",
)

APPROACHES = ["normal_region_growing", *FUZZY_CONNECTEDNESS_VARIANTS]

CONFIG = load_config_json(str(CONFIG_PATH), {})
CONFIG["DOWNSCALE_FACTORS"] = list(DOWNSCALE_FACTORS)
CONFIG["LOAD_CACHE"] = False
CONFIG["SAVE_CACHE"] = False
RUN_CONFIG = scale_config_to_resolution(copy.deepcopy(CONFIG))

train_ids, val_ids, test_ids, all_ids = get_data_splits(str(BASE_PATH))
SAMPLE_RECORDS = [
    {"img_id": img_id, "split": "train"} for img_id in train_ids[:TRAIN_SAMPLE_SIZE]
] + [
    {"img_id": img_id, "split": "val"} for img_id in val_ids[:VAL_SAMPLE_SIZE]
]
SAMPLE_IMAGE_IDS = [record["img_id"] for record in SAMPLE_RECORDS]
SAMPLE_SPLITS_BY_ID = {record["img_id"]: record["split"] for record in SAMPLE_RECORDS}

config_df = pd.DataFrame([
    {
        "train_sample_size": TRAIN_SAMPLE_SIZE,
        "val_sample_size": VAL_SAMPLE_SIZE,
        "sample_image_ids": SAMPLE_IMAGE_IDS,
        "approaches": APPROACHES,
        "candidate_min_vesselness": CANDIDATE_MIN_VESSELNESS,
        "max_processed_voxels": MAX_PROCESSED_VOXELS,
        "max_candidate_voxels": MAX_CANDIDATE_VOXELS,
        "use_local_multi_seeds": USE_LOCAL_MULTI_SEEDS,
        "seed_search_radius": SEED_SEARCH_RADIUS,
        "max_seeds_per_ostium": MAX_SEEDS_PER_OSTIUM,
        "seed_min_vesselness": SEED_MIN_VESSELNESS,
        "min_seed_distance_voxels": MIN_SEED_DISTANCE_VOXELS,
        "use_background_seeds": USE_BACKGROUND_SEEDS,
        "fuzzy_connectedness_variants": FUZZY_CONNECTEDNESS_VARIANTS,
        "base_path": str(BASE_PATH),
        "downscale_factors": DOWNSCALE_FACTORS,
        "load_cache": RUN_CONFIG["LOAD_CACHE"],
        "save_cache": RUN_CONFIG["SAVE_CACHE"],
    }
])
config_df


,train_sample_size,val_sample_size,sample_image_ids,approaches,candidate_min_vesselness,max_processed_voxels,max_candidate_voxels,use_local_multi_seeds,seed_search_radius,max_seeds_per_ostium,seed_min_vesselness,min_seed_distance_voxels,use_background_seeds,fuzzy_connectedness_variants,base_path,downscale_factors,load_cache,save_cache
0,30,0,"[28, 380, 854, 937, 315, 428, 752, 13, 538, 96...","[normal_region_growing, fc_wp_seed_s100_r2_m4_...",0.01,500000,500000,True,3,8,0.02,1.0,False,{'fc_wp_seed_s100_r2_m4_seed020_n18': {'alpha'...,/media/matheus/HD/DatasetsCCTA/ImageCAS/1-1000,"(2, 2, 1)",False,False


## 2.1. Parâmetros por abordagem

Tabela compacta das variantes testadas nesta rodada.


In [4]:
approach_params_df = pd.DataFrame(
    [
        {
            "approach": approach,
            **params,
        }
        for approach, params in FUZZY_CONNECTEDNESS_VARIANTS.items()
    ]
)

visible_param_columns = [
    "approach",
    "variant_group",
    "segment_per_ostium",
    "alpha",
    "sigma_hu",
    "candidate_min_vesselness",
    "neighborhood",
    "vesselness_affinity_mode",
    "vesselness_power",
    "vesselness_floor",
    "edge_affinity_mode",
    "vesselness_weight",
    "seed_search_radius",
    "max_seeds_per_ostium",
    "seed_min_vesselness",
    "postprocess_closing_radius",
    "postprocess_dilation_radius",
]
approach_params_df.reindex(columns=visible_param_columns)




,approach,variant_group,segment_per_ostium,alpha,sigma_hu,candidate_min_vesselness,neighborhood,vesselness_affinity_mode,vesselness_power,vesselness_floor,edge_affinity_mode,vesselness_weight,seed_search_radius,max_seeds_per_ostium,seed_min_vesselness,postprocess_closing_radius,postprocess_dilation_radius
0,fc_wp_seed_s100_r2_m4_seed020_n18,selected_top5_30_images,NaN,0.16,100.0,0.02,18,geometric_mean,1.0,0.02,weighted_product,0.9,2.0,4.0,0.02,NaN,NaN
1,fc_wp_seed_s100_r2_m6_seed020_n18,selected_top5_30_images,NaN,0.16,100.0,0.02,18,geometric_mean,1.0,0.02,weighted_product,0.9,2.0,6.0,0.02,NaN,NaN
2,fc_wp_seed_s100_r3_m4_seed020_n18,selected_top5_30_images,NaN,0.16,100.0,0.02,18,geometric_mean,1.0,0.02,weighted_product,0.9,3.0,4.0,0.02,NaN,NaN
3,fc_wp_seed_s100_r2_m8_seed020_n18,selected_top5_30_images,NaN,0.16,100.0,0.02,18,geometric_mean,1.0,0.02,weighted_product,0.9,2.0,8.0,0.02,NaN,NaN
4,fc_wp_alpha_a160_s100_w090_c020_n18,selected_top5_anchor,NaN,0.16,100.0,0.02,18,geometric_mean,1.0,0.02,weighted_product,0.9,NaN,NaN,NaN,NaN,NaN


## 3. Funções auxiliares

Inclui carregamento, pré-processamento, detecção dos óstios, baseline de region growing e fuzzy connectedness.


In [5]:
def load_sample_image(img_id: int) -> dict:
    """Carrega imagem, label e metadados reduzidos de um caso."""
    img_path = BASE_PATH / f"{img_id}.img.nii.gz"
    label_path = BASE_PATH / f"{img_id}.label.nii.gz"
    nii_img, nii_label = load_raw_img_and_label(str(img_path), str(label_path))
    image = nii_img.get_fdata(dtype=np.float32)
    label = nii_label.get_fdata(dtype=np.float32).astype(np.uint8)
    spacing = tuple(float(value) for value in nii_img.header.get_zooms()[:3])
    down_label = downscale_image_ndi(label, DOWNSCALE_FACTORS, order=0).astype(np.uint8)
    scaled_spacing = tuple(
        spacing[idx] * DOWNSCALE_FACTORS[idx] for idx in range(len(DOWNSCALE_FACTORS))
    )
    return {
        "img_id": img_id,
        "image": image,
        "down_label": down_label,
        "spacing": spacing,
        "scaled_spacing": scaled_spacing,
        "image_shape": image.shape,
        "label_shape": label.shape,
        "down_label_shape": down_label.shape,
    }


def build_lcc_input(image: np.ndarray) -> dict:
    """Gera o volume reduzido, threshold normal e maior componente conectada."""
    down_image = downscale_image_ndi(image, DOWNSCALE_FACTORS, order=3).astype(np.float32)
    max_hu = float(np.percentile(down_image, RUN_CONFIG["MAX_THRESHOLD_PERCENTILE"]))
    _, threshold_mask, _ = threshold_image_with_offset(
        down_image,
        min_val=RUN_CONFIG.get("MIN_THRESHOLD", MIN_HU),
        max_val=int(max_hu),
    )
    lcc_image, lcc_mask = build_lcc_image_from_mask(
        down_image,
        threshold_mask,
        offset=abs(int(RUN_CONFIG.get("MIN_THRESHOLD", MIN_HU))),
        per_slice=True,
    )
    return {
        "down_image": down_image,
        "lcc_image": lcc_image,
        "lcc_mask": lcc_mask.astype(bool),
        "max_hu": max_hu,
        "threshold_voxels": int(threshold_mask.sum()),
        "lcc_voxels": int(lcc_mask.sum()),
    }


In [6]:
def run_detection_stage(img_id: int, sample: dict, lcc_data: dict) -> dict:
    """Executa vesselness de óstios, círculos, aorta e avaliação dos óstios."""
    stage_root = BASE_SAVE_PATH / str(img_id)
    lcc_image = lcc_data["lcc_image"]
    scaled_spacing = sample["scaled_spacing"]
    vesselness_spacing = (scaled_spacing[1], scaled_spacing[0], scaled_spacing[2])

    vesselness_ostios = get_or_compute_vesselness(
        str(img_id),
        lcc_image,
        cache_dir=str(stage_root / "vesselness_ostios_cache"),
        vesselness_config=RUN_CONFIG["VESSELNESS_AORTA"],
        load_cache=RUN_CONFIG["LOAD_CACHE"],
        save_cache=RUN_CONFIG["SAVE_CACHE"],
        use_gpu=RUN_CONFIG.get("USE_GPU", False),
        spacing=vesselness_spacing,
    )
    detected_circles = get_or_detect_aorta_circles(
        str(img_id),
        lcc_image,
        DOWNSCALE_FACTORS,
        scaled_spacing,
        RUN_CONFIG["CIRCLE_DETECTION"],
        stage_root,
        load_cache=RUN_CONFIG["LOAD_CACHE"],
        save_cache=RUN_CONFIG["SAVE_CACHE"],
    )
    aorta_mask = get_or_segment_aorta(
        str(img_id),
        lcc_image,
        detected_circles,
        RUN_CONFIG["LEVEL_SET"],
        stage_root,
        load_cache=RUN_CONFIG["LOAD_CACHE"],
        save_cache=RUN_CONFIG["SAVE_CACHE"],
        use_gpu=RUN_CONFIG.get("USE_GPU", False),
    )

    result = {
        "img_id": img_id,
        "sample_split": SAMPLE_SPLITS_BY_ID.get(img_id),
        "ostia_found": False,
        "ostia_status": "not_found",
        "ostia_acceptable": False,
        "both_correct": False,
        "both_tolerable": False,
        "left_dist_mm": np.inf,
        "right_dist_mm": np.inf,
        "ostia_left": None,
        "ostia_right": None,
        "label_artery": (sample["down_label"] == 1).astype(np.uint8),
        "num_circles": len(detected_circles),
        "aorta_voxels": int(aorta_mask.sum()),
        "ostia_error": None,
    }

    try:
        ostia_eval = detect_and_evaluate_ostia(
            aorta_mask,
            vesselness_ostios,
            sample["down_label"],
            scaled_spacing,
            RUN_CONFIG,
        )
    except ValueError as exc:
        result["ostia_error"] = str(exc)
        return result

    if ostia_eval["ostia_left"] is None or ostia_eval["ostia_right"] is None:
        result["ostia_error"] = "Ostio esquerdo ou direito nao encontrado."
        return result

    if ostia_eval["both_correct"]:
        ostia_status = "both_correct"
    elif ostia_eval["both_tolerable"]:
        ostia_status = "both_tolerable"
    else:
        ostia_status = "found_but_wrong"

    result.update(
        {
            "ostia_found": True,
            "ostia_status": ostia_status,
            "ostia_acceptable": bool(ostia_eval["both_correct"] or ostia_eval["both_tolerable"]),
            "both_correct": bool(ostia_eval["both_correct"]),
            "both_tolerable": bool(ostia_eval["both_tolerable"]),
            "left_dist_mm": ostia_eval["left_info"]["physical_dist"],
            "right_dist_mm": ostia_eval["right_info"]["physical_dist"],
            "ostia_left": tuple(map(int, ostia_eval["ostia_left"])),
            "ostia_right": tuple(map(int, ostia_eval["ostia_right"])),
            "label_artery": ostia_eval["label_artery"],
        }
    )
    return result


def compute_artery_vesselness(img_id: int, lcc_image: np.ndarray, scaled_spacing) -> np.ndarray:
    """Calcula vesselness arterial uma vez por imagem."""
    vesselness_spacing = (scaled_spacing[1], scaled_spacing[0], scaled_spacing[2])
    return get_or_compute_vesselness(
        str(img_id),
        lcc_image,
        cache_dir=str(BASE_SAVE_PATH / str(img_id) / "vesselness_artery_cache"),
        vesselness_config=RUN_CONFIG["VESSELNESS_ARTERY"],
        load_cache=RUN_CONFIG["LOAD_CACHE"],
        save_cache=RUN_CONFIG["SAVE_CACHE"],
        use_gpu=RUN_CONFIG.get("USE_GPU", False),
        spacing=vesselness_spacing,
    )


def run_segmentation_approaches(img_id: int, sample: dict, lcc_data: dict, detection: dict) -> pd.DataFrame:
    """Executa baseline normal e fuzzy connectedness para uma imagem."""
    if not detection["ostia_found"]:
        return pd.DataFrame(
            [
                {
                    "img_id": img_id,
                    "sample_split": SAMPLE_SPLITS_BY_ID.get(img_id),
                    "approach": approach,
                    "segmentation_attempted": False,
                    "dice_artery": 0.0,
                    "artery_voxels": 0,
                    "connectivity_max": np.nan,
                    "processed_voxels": 0,
                    "segmentation_error": detection["ostia_error"],
                }
                for approach in APPROACHES
            ]
        )

    lcc_image = lcc_data["lcc_image"]
    vesselness_artery = compute_artery_vesselness(
        img_id,
        lcc_image,
        sample["scaled_spacing"],
    )
    vesselness_norm = normalize_vesselness(vesselness_artery)
    base_object_seeds = [detection["ostia_left"], detection["ostia_right"]]

    def build_connectedness_inputs(
        candidate_min_vesselness: float,
        use_background_seeds: bool,
        *,
        seed_search_radius: int = SEED_SEARCH_RADIUS,
        max_seeds_per_ostium: int = MAX_SEEDS_PER_OSTIUM,
        seed_min_vesselness: float = SEED_MIN_VESSELNESS,
        min_seed_distance_voxels: float = MIN_SEED_DISTANCE_VOXELS,
    ) -> tuple[np.ndarray, dict, list[tuple[int, int, int]], dict, list[tuple[int, int, int]] | None]:
        candidate_mask = (lcc_data["lcc_mask"] > 0) & (vesselness_norm >= float(candidate_min_vesselness))
        candidate_mask, candidate_details = limit_candidate_mask_by_vesselness(
            candidate_mask,
            vesselness_norm,
            MAX_CANDIDATE_VOXELS,
            min_candidate_vesselness=float(candidate_min_vesselness),
        )
        if USE_LOCAL_MULTI_SEEDS:
            object_seeds, seed_details = collect_local_object_seeds(
                base_object_seeds,
                vesselness_norm,
                candidate_mask,
                search_radius=seed_search_radius,
                max_seeds_per_ostium=max_seeds_per_ostium,
                min_seed_vesselness=seed_min_vesselness,
                min_seed_distance_voxels=min_seed_distance_voxels,
            )
        else:
            object_seeds = []
            for seed in base_object_seeds:
                coord = valid_seed(seed, candidate_mask.shape, candidate_mask=candidate_mask)
                if coord is not None:
                    object_seeds.append(coord)
            seed_details = {
                "object_seed_count": len(object_seeds),
                "object_seed_counts_per_ostium": [1 if seed is not None else 0 for seed in base_object_seeds],
            }
        seed_details.update(
            {
                "seed_search_radius": int(seed_search_radius),
                "max_seeds_per_ostium": int(max_seeds_per_ostium),
                "seed_min_vesselness": float(seed_min_vesselness),
                "min_seed_distance_voxels": float(min_seed_distance_voxels),
            }
        )

        background_seeds = build_background_seeds(candidate_mask, vesselness_artery) if use_background_seeds else None
        return candidate_mask, candidate_details, object_seeds, seed_details, background_seeds

    candidate_mask, candidate_details, object_seeds, seed_details, background_seeds = build_connectedness_inputs(
        CANDIDATE_MIN_VESSELNESS,
        USE_BACKGROUND_SEEDS,
    )

    rows = []
    label_artery = detection["label_artery"]

    raw_normal = normal_region_growing_from_ostia(
        vesselness_artery,
        detection["ostia_left"],
        detection["ostia_right"],
        RUN_CONFIG,
    )
    normal_mask = postprocess_artery_mask(raw_normal, RUN_CONFIG)
    rows.append(
        {
            "img_id": img_id,
            "sample_split": SAMPLE_SPLITS_BY_ID.get(img_id),
            "approach": "normal_region_growing",
            "variant_group": None,
            "segmentation_attempted": True,
            "dice_artery": float(dice_score(normal_mask, label_artery)),
            "artery_voxels": int(normal_mask.sum()),
            "raw_artery_voxels": int(raw_normal.sum()),
            "connectivity_max": np.nan,
            "processed_voxels": np.nan,
            **candidate_details,
            **seed_details,
            "alpha": np.nan,
            "sigma_hu": np.nan,
            "neighborhood": np.nan,
            "vesselness_affinity_mode": None,
            "vesselness_power": np.nan,
            "vesselness_floor": np.nan,
            "edge_affinity_mode": None,
            "vesselness_weight": np.nan,
            "mask_strategy": None,
            "connectivity_percentile": np.nan,
            "effective_alpha": np.nan,
            "uses_background_seeds": False,
            "background_seed_count": 0,
            "segment_per_ostium": False,
            "seed_search_radius": SEED_SEARCH_RADIUS,
            "max_seeds_per_ostium": MAX_SEEDS_PER_OSTIUM,
            "seed_min_vesselness": SEED_MIN_VESSELNESS,
            "postprocess_closing_radius": RUN_CONFIG["POSTPROCESSING"]["closing_radius"],
            "postprocess_dilation_radius": RUN_CONFIG["POSTPROCESSING"]["dilation_radius"],
            "segmentation_error": None,
        }
    )

    def collect_seed_group_for_single_ostium(seed, candidate_mask: np.ndarray, params: dict) -> tuple[list[tuple[int, int, int]], int]:
        if USE_LOCAL_MULTI_SEEDS:
            seeds, details = collect_local_object_seeds(
                [seed],
                vesselness_norm,
                candidate_mask,
                search_radius=params.get("seed_search_radius", SEED_SEARCH_RADIUS),
                max_seeds_per_ostium=params.get("max_seeds_per_ostium", MAX_SEEDS_PER_OSTIUM),
                min_seed_vesselness=params.get("seed_min_vesselness", SEED_MIN_VESSELNESS),
                min_seed_distance_voxels=params.get("min_seed_distance_voxels", MIN_SEED_DISTANCE_VOXELS),
            )
            return seeds, int(details["object_seed_count"])
        coord = valid_seed(seed, candidate_mask.shape, candidate_mask=candidate_mask)
        return ([coord], 1) if coord is not None else ([], 0)

    def run_fc_once(seeds, params, candidate_mask, background_seeds):
        return fuzzy_connectedness_segmentation(
            lcc_image,
            vesselness_artery,
            seeds,
            alpha=params["alpha"],
            sigma_hu=params["sigma_hu"],
            neighborhood=params["neighborhood"],
            candidate_mask=candidate_mask,
            background_seeds=background_seeds,
            max_processed_voxels=MAX_PROCESSED_VOXELS,
            vesselness_affinity_mode=params.get("vesselness_affinity_mode", "mean"),
            vesselness_power=params.get("vesselness_power", 1.0),
            vesselness_floor=params.get("vesselness_floor", 0.0),
            edge_affinity_mode=params.get("edge_affinity_mode", "min"),
            vesselness_weight=params.get("vesselness_weight", 0.7),
            mask_strategy=params.get("mask_strategy", "alpha"),
            connectivity_percentile=params.get("connectivity_percentile"),
        )

    for approach, params in FUZZY_CONNECTEDNESS_VARIANTS.items():
        variant_candidate_mask, variant_candidate_details, variant_object_seeds, variant_seed_details, variant_background_seeds = build_connectedness_inputs(
            params.get("candidate_min_vesselness", CANDIDATE_MIN_VESSELNESS),
            params.get("use_background_seeds", USE_BACKGROUND_SEEDS),
            seed_search_radius=params.get("seed_search_radius", SEED_SEARCH_RADIUS),
            max_seeds_per_ostium=params.get("max_seeds_per_ostium", MAX_SEEDS_PER_OSTIUM),
            seed_min_vesselness=params.get("seed_min_vesselness", SEED_MIN_VESSELNESS),
            min_seed_distance_voxels=params.get("min_seed_distance_voxels", MIN_SEED_DISTANCE_VOXELS),
        )
        segment_per_ostium = bool(params.get("segment_per_ostium", False))

        if segment_per_ostium:
            raw_masks = []
            detail_items = []
            per_ostium_counts = []
            for seed in base_object_seeds:
                single_seeds, seed_count = collect_seed_group_for_single_ostium(seed, variant_candidate_mask, params)
                per_ostium_counts.append(seed_count)
                if not single_seeds:
                    continue
                fc_result = run_fc_once(single_seeds, params, variant_candidate_mask, variant_background_seeds)
                raw_masks.append(fc_result["mask"] > 0)
                detail_items.append(fc_result["details"])

            raw_mask = (
                np.logical_or.reduce(raw_masks).astype(np.uint8)
                if raw_masks
                else np.zeros_like(lcc_image, dtype=np.uint8)
            )
            if detail_items:
                effective_alphas = [item["effective_alpha"] for item in detail_items if np.isfinite(item["effective_alpha"])]
                details = {
                    **detail_items[0],
                    "processed_voxels": int(sum(item["processed_voxels"] for item in detail_items)),
                    "max_connectivity": float(max(item["max_connectivity"] for item in detail_items)),
                    "effective_alpha": float(np.mean(effective_alphas)) if effective_alphas else np.nan,
                    "background_processed_voxels": int(sum(
                        item["background_processed_voxels"] or 0 for item in detail_items
                    )),
                }
            else:
                details = {
                    "max_connectivity": 0.0,
                    "processed_voxels": 0,
                    "alpha": float(params["alpha"]),
                    "sigma_hu": float(params["sigma_hu"]),
                    "neighborhood": int(params["neighborhood"]),
                    "vesselness_affinity_mode": params.get("vesselness_affinity_mode", "mean"),
                    "vesselness_power": float(params.get("vesselness_power", 1.0)),
                    "vesselness_floor": float(params.get("vesselness_floor", 0.0)),
                    "edge_affinity_mode": params.get("edge_affinity_mode", "min"),
                    "vesselness_weight": float(params.get("vesselness_weight", 0.7)),
                    "mask_strategy": params.get("mask_strategy", "alpha"),
                    "connectivity_percentile": params.get("connectivity_percentile"),
                    "effective_alpha": np.nan,
                    "uses_background_seeds": bool(variant_background_seeds),
                    "background_seed_count": len(variant_background_seeds or []),
                }
            variant_seed_details = {
                "object_seed_count": int(sum(per_ostium_counts)),
                "object_seed_counts_per_ostium": per_ostium_counts,
            }
        else:
            fc_result = run_fc_once(
                variant_object_seeds,
                params,
                variant_candidate_mask,
                variant_background_seeds,
            )
            raw_mask = fc_result["mask"]
            details = fc_result["details"]

        artery_mask = postprocess_artery_mask(
            raw_mask,
            RUN_CONFIG,
            closing_radius=params.get("postprocess_closing_radius"),
            dilation_radius=params.get("postprocess_dilation_radius"),
        )
        rows.append(
            {
                "img_id": img_id,
                "sample_split": SAMPLE_SPLITS_BY_ID.get(img_id),
                "approach": approach,
                "variant_group": params.get("variant_group"),
                "segmentation_attempted": True,
                "dice_artery": float(dice_score(artery_mask, label_artery)),
                "artery_voxels": int(artery_mask.sum()),
                "raw_artery_voxels": int(raw_mask.sum()),
                "connectivity_max": details["max_connectivity"],
                "processed_voxels": details["processed_voxels"],
                **variant_candidate_details,
                **variant_seed_details,
                "alpha": details["alpha"],
                "sigma_hu": details["sigma_hu"],
                "neighborhood": details["neighborhood"],
                "vesselness_affinity_mode": details["vesselness_affinity_mode"],
                "vesselness_power": details["vesselness_power"],
                "vesselness_floor": details["vesselness_floor"],
                "edge_affinity_mode": details["edge_affinity_mode"],
                "vesselness_weight": details["vesselness_weight"],
                "mask_strategy": details["mask_strategy"],
                "connectivity_percentile": details["connectivity_percentile"],
                "effective_alpha": details["effective_alpha"],
                "uses_background_seeds": details["uses_background_seeds"],
                "background_seed_count": details["background_seed_count"],
                "segment_per_ostium": segment_per_ostium,
                "seed_search_radius": variant_seed_details.get("seed_search_radius", params.get("seed_search_radius", SEED_SEARCH_RADIUS)),
                "max_seeds_per_ostium": variant_seed_details.get("max_seeds_per_ostium", params.get("max_seeds_per_ostium", MAX_SEEDS_PER_OSTIUM)),
                "seed_min_vesselness": variant_seed_details.get("seed_min_vesselness", params.get("seed_min_vesselness", SEED_MIN_VESSELNESS)),
                "postprocess_closing_radius": params.get("postprocess_closing_radius", RUN_CONFIG["POSTPROCESSING"]["closing_radius"]),
                "postprocess_dilation_radius": params.get("postprocess_dilation_radius", RUN_CONFIG["POSTPROCESSING"]["dilation_radius"]),
                "segmentation_error": None,
            }
        )

    return pd.DataFrame(rows)


def run_sample_experiment(img_id: int) -> dict:
    sample = load_sample_image(img_id)
    lcc_data = build_lcc_input(sample["image"])
    detection = run_detection_stage(img_id, sample, lcc_data)
    segmentation_df = run_segmentation_approaches(img_id, sample, lcc_data, detection)

    sample_info_df = pd.DataFrame([
        {
            "img_id": img_id,
            "sample_split": SAMPLE_SPLITS_BY_ID.get(img_id),
            "image_shape": sample["image_shape"],
            "label_shape": sample["label_shape"],
            "down_label_shape": sample["down_label_shape"],
            "spacing_mm": sample["spacing"],
            "scaled_spacing_mm": sample["scaled_spacing"],
            "max_hu": lcc_data["max_hu"],
            "threshold_voxels": lcc_data["threshold_voxels"],
            "lcc_voxels": lcc_data["lcc_voxels"],
        }
    ])

    ostia_df = pd.DataFrame([
        {
            key: value
            for key, value in detection.items()
            if key != "label_artery"
        }
    ])
    return {
        "sample_info_df": sample_info_df,
        "ostia_df": ostia_df,
        "segmentation_df": segmentation_df,
    }


def concat_outputs(outputs: list[dict], key: str) -> pd.DataFrame:
    frames = [output[key] for output in outputs if not output[key].empty]
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()








## 4. Amostras

Lista de imagens usadas nesta rodada.


In [7]:
sample_ids_df = pd.DataFrame(
    [
        {
            "sample_order": idx,
            "img_id": record["img_id"],
            "split": record["split"],
        }
        for idx, record in enumerate(SAMPLE_RECORDS, start=1)
    ]
)
sample_ids_df



,sample_order,img_id,split
0,1,28,train
1,2,380,train
2,3,854,train
3,4,937,train
4,5,315,train
5,6,428,train
6,7,752,train
7,8,13,train
8,9,538,train
9,10,965,train


## 5. Execução

Esta célula executa detecção dos óstios uma vez por imagem, calcula a vesselness arterial uma vez e compara o baseline com fuzzy connectedness.


In [8]:
sample_outputs = [run_sample_experiment(img_id) for img_id in SAMPLE_IMAGE_IDS]

sample_info_df = concat_outputs(sample_outputs, "sample_info_df")
ostia_df = concat_outputs(sample_outputs, "ostia_df")
segmentation_df = concat_outputs(sample_outputs, "segmentation_df")

segmentation_df = segmentation_df.merge(
    ostia_df[
        [
            "img_id",
            "sample_split",
            "ostia_status",
            "ostia_found",
            "ostia_acceptable",
            "both_correct",
            "both_tolerable",
            "left_dist_mm",
            "right_dist_mm",
        ]
    ],
    on=["img_id", "sample_split"],
    how="left",
)

segmentation_df.sort_values(["img_id", "approach"])



Parada na fatia 154: Δr=0.00 ou dist=46.62
Parada na fatia 128: Δr=0.00 ou dist=37.20
Parada na fatia 101: Δr=6.33 ou dist=50.84
Parada na fatia 192: Δr=1.00 ou dist=33.62
Parada na fatia 140: Δr=10.00 ou dist=34.99
Parada na fatia 117: Δr=1.00 ou dist=45.04
Parada na fatia 161: Δr=1.00 ou dist=32.20
Parada na fatia 173: Δr=5.00 ou dist=33.84
Parada na fatia 165: Δr=1.00 ou dist=64.64
Parada na fatia 182: Δr=3.00 ou dist=42.45
Parada na fatia 152: Δr=1.50 ou dist=34.51
Parada na fatia 160: Δr=0.00 ou dist=36.07
Parada na fatia 137: Δr=9.00 ou dist=46.32
Parada na fatia 124: Δr=0.00 ou dist=32.65
Parada na fatia 134: Δr=0.50 ou dist=35.13
Parada na fatia 149: Δr=0.50 ou dist=30.54
Parada na fatia 150: Δr=1.00 ou dist=34.31
Parada na fatia 147: Δr=7.00 ou dist=45.49
Parada na fatia 177: Δr=0.50 ou dist=36.06
Parada na fatia 141: Δr=2.00 ou dist=31.02
Parada na fatia 172: Δr=0.00 ou dist=34.53
Parada na fatia 139: Δr=4.00 ou dist=47.54
Parada na fatia 117: Δr=0.00 ou dist=34.00
Parada na 

,img_id,sample_split,approach,variant_group,segmentation_attempted,dice_artery,artery_voxels,raw_artery_voxels,connectivity_max,processed_voxels,...,postprocess_closing_radius,postprocess_dilation_radius,segmentation_error,ostia_status,ostia_found,ostia_acceptable,both_correct,both_tolerable,left_dist_mm,right_dist_mm
47,13,train,fc_wp_alpha_a160_s100_w090_c020_n18,selected_top5_anchor,True,0.559382,24449,6631,1.0,6631.0,...,3,2,None,both_tolerable,True,True,False,True,0.000000,1.103140
43,13,train,fc_wp_seed_s100_r2_m4_seed020_n18,selected_top5_30_images,True,0.559382,24449,6631,1.0,6631.0,...,3,2,None,both_tolerable,True,True,False,True,0.000000,1.103140
44,13,train,fc_wp_seed_s100_r2_m6_seed020_n18,selected_top5_30_images,True,0.559382,24449,6631,1.0,6631.0,...,3,2,None,both_tolerable,True,True,False,True,0.000000,1.103140
46,13,train,fc_wp_seed_s100_r2_m8_seed020_n18,selected_top5_30_images,True,0.559382,24449,6631,1.0,6631.0,...,3,2,None,both_tolerable,True,True,False,True,0.000000,1.103140
45,13,train,fc_wp_seed_s100_r3_m4_seed020_n18,selected_top5_30_images,True,0.559382,24449,6631,1.0,6631.0,...,3,2,None,both_tolerable,True,True,False,True,0.000000,1.103140
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
55,965,train,fc_wp_seed_s100_r2_m4_seed020_n18,selected_top5_30_images,True,0.516902,18485,4880,1.0,4880.0,...,3,2,None,both_tolerable,True,True,False,True,1.482029,5.416548
56,965,train,fc_wp_seed_s100_r2_m6_seed020_n18,selected_top5_30_images,True,0.516902,18485,4880,1.0,4880.0,...,3,2,None,both_tolerable,True,True,False,True,1.482029,5.416548
58,965,train,fc_wp_seed_s100_r2_m8_seed020_n18,selected_top5_30_images,True,0.516611,18511,4882,1.0,4882.0,...,3,2,None,both_tolerable,True,True,False,True,1.482029,5.416548
57,965,train,fc_wp_seed_s100_r3_m4_seed020_n18,selected_top5_30_images,True,0.516902,18485,4880,1.0,4880.0,...,3,2,None,both_tolerable,True,True,False,True,1.482029,5.416548


## 6. Óstios

Status da detecção dos óstios, usado para interpretar o Dice da segmentação.


In [9]:
ostia_df.sort_values(["sample_split", "img_id"])[
    [
        "img_id",
        "sample_split",
        "ostia_status",
        "ostia_found",
        "ostia_acceptable",
        "both_correct",
        "both_tolerable",
        "left_dist_mm",
        "right_dist_mm",
        "num_circles",
        "aorta_voxels",
        "ostia_error",
    ]
]



,img_id,sample_split,ostia_status,ostia_found,ostia_acceptable,both_correct,both_tolerable,left_dist_mm,right_dist_mm,num_circles,aorta_voxels,ostia_error
7,13,train,both_tolerable,True,True,False,True,0.000000,1.103140,67,121815,None
0,28,train,both_tolerable,True,True,False,True,1.073700,0.837506,120,190293,None
14,44,train,both_tolerable,True,True,False,True,0.589844,0.000000,71,130342,None
21,90,train,both_tolerable,True,True,False,True,1.320312,0.500000,135,297595,None
15,129,train,both_tolerable,True,True,False,True,1.492188,0.500000,125,218138,None
24,175,train,both_tolerable,True,True,False,True,0.900456,0.500000,105,281351,None
27,194,train,both_tolerable,True,True,False,True,0.809574,1.739843,118,293039,None
28,305,train,both_tolerable,True,True,False,True,2.526628,0.000000,104,163889,None
4,315,train,both_tolerable,True,True,False,True,0.000000,0.846936,134,385823,None
20,330,train,both_tolerable,True,True,False,True,0.000000,1.974512,102,283088,None


## 7. Dice por imagem

Tabela larga para comparar o baseline e as variantes fuzzy connectedness caso a caso.


In [10]:
dice_by_image_df = (
    segmentation_df.pivot_table(
        index=["sample_split", "img_id", "ostia_status"],
        columns="approach",
        values="dice_artery",
        aggfunc="first",
    )
    .reset_index()
    .sort_values(["sample_split", "img_id"])
)
dice_by_image_df.columns.name = None
dice_by_image_df



,sample_split,img_id,ostia_status,fc_wp_alpha_a160_s100_w090_c020_n18,fc_wp_seed_s100_r2_m4_seed020_n18,fc_wp_seed_s100_r2_m6_seed020_n18,fc_wp_seed_s100_r2_m8_seed020_n18,fc_wp_seed_s100_r3_m4_seed020_n18,normal_region_growing
0,train,13,both_tolerable,0.559382,0.559382,0.559382,0.559382,0.559382,0.571285
1,train,28,both_tolerable,0.657512,0.657512,0.657512,0.657512,0.657512,0.672063
2,train,44,both_tolerable,0.479649,0.479649,0.479649,0.479649,0.479649,0.360994
3,train,90,both_tolerable,0.555233,0.555233,0.555233,0.555233,0.555233,0.694632
4,train,129,both_tolerable,0.712886,0.712886,0.712886,0.712886,0.712886,0.716542
5,train,175,both_tolerable,0.530249,0.530249,0.530249,0.530249,0.530249,0.460065
6,train,194,both_tolerable,0.797573,0.797573,0.797573,0.797573,0.797573,0.790273
7,train,305,both_tolerable,0.678696,0.678696,0.678696,0.678696,0.678696,0.687888
8,train,315,both_tolerable,0.644687,0.644687,0.644687,0.644687,0.644687,0.648588
9,train,330,both_tolerable,0.598486,0.598486,0.598486,0.598486,0.598486,0.617252


## 8. Resumo

Resumo médio por abordagem, incluindo Dice e qualidade da detecção dos óstios.


In [11]:
summary_df = (
    segmentation_df.groupby("approach", as_index=False)
    .agg(
        mean_dice=("dice_artery", "mean"),
        std_dice=("dice_artery", "std"),
        min_dice=("dice_artery", "min"),
        max_dice=("dice_artery", "max"),
        evaluated_images=("img_id", "nunique"),
        ostia_found_rate=("ostia_found", "mean"),
        ostia_acceptable_rate=("ostia_acceptable", "mean"),
        both_correct_rate=("both_correct", "mean"),
        both_tolerable_rate=("both_tolerable", "mean"),
        mean_artery_voxels=("artery_voxels", "mean"),
        mean_raw_artery_voxels=("raw_artery_voxels", "mean"),
        mean_processed_voxels=("processed_voxels", "mean"),
    )
)
summary_df["analysis_score"] = summary_df["mean_dice"] * summary_df["ostia_acceptable_rate"]
summary_df = summary_df.sort_values(
    ["analysis_score", "mean_dice", "ostia_acceptable_rate"],
    ascending=False,
).reset_index(drop=True)
summary_df



,approach,mean_dice,std_dice,min_dice,max_dice,evaluated_images,ostia_found_rate,ostia_acceptable_rate,both_correct_rate,both_tolerable_rate,mean_artery_voxels,mean_raw_artery_voxels,mean_processed_voxels,analysis_score
0,fc_wp_seed_s100_r2_m4_seed020_n18,0.606715,0.118613,0.198589,0.797573,30,1.0,0.9,0.0,0.9,28710.266667,7781.266667,7781.266667,0.546043
1,fc_wp_seed_s100_r2_m6_seed020_n18,0.606715,0.118613,0.198589,0.797573,30,1.0,0.9,0.0,0.9,28710.266667,7781.266667,7781.266667,0.546043
2,fc_wp_seed_s100_r3_m4_seed020_n18,0.606715,0.118613,0.198589,0.797573,30,1.0,0.9,0.0,0.9,28710.266667,7781.266667,7781.266667,0.546043
3,fc_wp_seed_s100_r2_m8_seed020_n18,0.606705,0.118621,0.198589,0.797573,30,1.0,0.9,0.0,0.9,28711.133333,7781.333333,7781.333333,0.546035
4,fc_wp_alpha_a160_s100_w090_c020_n18,0.606673,0.118646,0.198589,0.797573,30,1.0,0.9,0.0,0.9,28714.000000,7781.633333,7781.633333,0.546006
5,normal_region_growing,0.598550,0.136364,0.174146,0.790273,30,1.0,0.9,0.0,0.9,34269.000000,8316.900000,NaN,0.538695


## 8.1. Resumo por Grupo

Agrupa as variantes para identificar quais famílias de parâmetros estão funcionando melhor.


In [12]:
group_summary_df = (
    segmentation_df[segmentation_df["approach"] != "normal_region_growing"]
    .groupby("variant_group", as_index=False)
    .agg(
        mean_dice=("dice_artery", "mean"),
        std_dice=("dice_artery", "std"),
        min_dice=("dice_artery", "min"),
        max_dice=("dice_artery", "max"),
        variants=("approach", "nunique"),
        evaluated_images=("img_id", "nunique"),
    )
    .sort_values(["mean_dice", "min_dice"], ascending=False)
    .reset_index(drop=True)
)
group_summary_df



,variant_group,mean_dice,std_dice,min_dice,max_dice,variants,evaluated_images
0,selected_top5_30_images,0.606712,0.117110,0.198589,0.797573,4,30
1,selected_top5_anchor,0.606673,0.118646,0.198589,0.797573,1,30
